In [0]:
%pip install transformers torch kagglehub

In [0]:
import os
import pandas as pd
from transformers import pipeline

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("journeyyouyeonkim/microsoft-tesla-finance-news-articles2020-2024")

print("Path to dataset files:", path)

In [0]:
path = "/home/spark-47c96ea6-94d0-4624-84fd-5f/.cache/kagglehub/datasets/journeyyouyeonkim/microsoft-tesla-finance-news-articles2020-2024/versions/1"

tsla_path = f"{path}/tsla_articles.csv"
msft_path = f"{path}/msft_articles.csv"

tsla_raw_df = pd.read_csv(tsla_path)
msft_raw_df = pd.read_csv(msft_path)

print("TSLA cols:", tsla_raw_df.columns.tolist(), "rows:", len(tsla_raw_df))
print("MSFT cols:", msft_raw_df.columns.tolist(), "rows:", len(msft_raw_df))

tsla_raw_df.head()

In [0]:
def normalize_text_only_news(df: pd.DataFrame, ticker: str, source_name: str) -> pd.DataFrame:
    out = df.copy()

    out["published_at"] = pd.to_datetime(out["date"], errors="coerce")
    out["Date"] = out["published_at"].dt.normalize()
    out["Ticker"] = ticker
    out["headline"] = out["text"].fillna("").astype(str)
    out["summary"] = ""
    out["source"] = source_name
    out["url"] = ""

    out = out[[
        "published_at",
        "Date",
        "Ticker",
        "headline",
        "summary",
        "source",
        "url"
    ]].dropna(subset=["Date"]).drop_duplicates()

    return out.reset_index(drop=True)

tsla_news_df = normalize_text_only_news(tsla_raw_df, "TSLA", "kaggle_tsla_articles")
msft_news_df = normalize_text_only_news(msft_raw_df, "MSFT", "kaggle_msft_articles")

print("TSLA normalized rows:", len(tsla_news_df))
print("MSFT normalized rows:", len(msft_news_df))
tsla_news_df.head()

In [0]:
def normalize_text_only_news(df: pd.DataFrame, ticker: str, source_name: str) -> pd.DataFrame:
    out = df.copy()

    out["published_at"] = pd.to_datetime(out["date"], errors="coerce")
    out["Date"] = out["published_at"].dt.normalize()
    out["Ticker"] = ticker
    out["headline"] = out["text"].fillna("").astype(str)
    out["summary"] = ""
    out["source"] = source_name
    out["url"] = ""

    out = out[[
        "published_at",
        "Date",
        "Ticker",
        "headline",
        "summary",
        "source",
        "url"
    ]].dropna(subset=["Date"]).drop_duplicates()

    return out.reset_index(drop=True)

tsla_news_df = normalize_text_only_news(tsla_raw_df, "TSLA", "kaggle_tsla_articles")
msft_news_df = normalize_text_only_news(msft_raw_df, "MSFT", "kaggle_msft_articles")

print("TSLA normalized rows:", len(tsla_news_df))
print("MSFT normalized rows:", len(msft_news_df))
tsla_news_df.head()

In [0]:
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    truncation=True
)

def finbert_score(texts, batch_size=32):
    scores = []
    negs = []
    neus = []
    poss = []

    for i in range(0, len(texts), batch_size):
        batch = [str(x)[:512] for x in texts[i:i+batch_size]]
        preds = sentiment_pipe(batch)

        for p in preds:
            label = p["label"].lower()
            score = float(p["score"])

            if label == "positive":
                scores.append(score)
                negs.append(0.0)
                neus.append(0.0)
                poss.append(score)
            elif label == "negative":
                scores.append(-score)
                negs.append(score)
                neus.append(0.0)
                poss.append(0.0)
            else:
                scores.append(0.0)
                negs.append(0.0)
                neus.append(score)
                poss.append(0.0)

    return scores, negs, neus, poss

In [0]:
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    truncation=True
)

def finbert_score(texts, batch_size=32):
    scores = []
    negs = []
    neus = []
    poss = []

    for i in range(0, len(texts), batch_size):
        batch = [str(x)[:512] for x in texts[i:i+batch_size]]
        preds = sentiment_pipe(batch)

        for p in preds:
            label = p["label"].lower()
            score = float(p["score"])

            if label == "positive":
                scores.append(score)
                negs.append(0.0)
                neus.append(0.0)
                poss.append(score)
            elif label == "negative":
                scores.append(-score)
                negs.append(score)
                neus.append(0.0)
                poss.append(0.0)
            else:
                scores.append(0.0)
                negs.append(0.0)
                neus.append(score)
                poss.append(0.0)

    return scores, negs, neus, poss

In [0]:
def add_sentiment_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    scores, negs, neus, poss = finbert_score(out["headline"].fillna("").tolist())

    out["headline_sentiment"] = scores
    out["summary_sentiment"] = 0.0
    out["combined_sentiment"] = out["headline_sentiment"]

    out["sentiment_neg"] = negs
    out["sentiment_neu"] = neus
    out["sentiment_pos"] = poss

    return out

tsla_news_df = add_sentiment_features(tsla_news_df)
msft_news_df = add_sentiment_features(msft_news_df)

tsla_news_df.head()

In [0]:
aapl_news_df = spark.table("p2_universe_news_raw").toPandas()

# keep only AAPL rows from that table
aapl_news_df = aapl_news_df[aapl_news_df["Ticker"] == "AAPL"].copy()

combined_news_df = pd.concat(
    [aapl_news_df, tsla_news_df, msft_news_df],
    axis=0,
    ignore_index=True
)

combined_news_df = combined_news_df.drop_duplicates(
    subset=["Ticker", "published_at", "headline"]
).reset_index(drop=True)

print(combined_news_df["Ticker"].value_counts())
combined_news_df.head()

In [0]:
spark.createDataFrame(combined_news_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("p2_universe_news_raw")